### Ventures AI

Chat Bot To query data from [Y-Combinator Startup directory](https://www.ycombinator.com/companies)
> Data Source: https://github.com/yc-oss/api (open sourec Y Combinator companies API)

In [ ]:
!uv add -r requirements.txt

### LLM Evaluation

Read the ingested companies from Postgres (`ventures_db.yc_oss`, loaded by the `yc_oss_to_ventures_db` Kestra flow)
> Get 1/10th of all records for generating ground truths

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import pandas as pd
import psycopg

load_dotenv('../.env')
openai_client = OpenAI()

In [3]:
conn = psycopg.connect(
    host="localhost",
    port=5440,
    dbname="ventures_db",
    user="postgres",
    password="postgres",
)

In [2]:
# with conn:
df = pd.read_sql("""
    SELECT 
        company_id id,
        title company_name_desc,
        content
    FROM yc_oss_fulltext
    where random() < 0.1
    """
, conn)

# conn.close()

print(f"Loaded {len(df)} companies")
print(df.shape)
df.head()

Loaded 499 companies
(499, 3)


/var/folders/m4/y4ly49q978l8dsj7_7n8ts_40000gp/T/ipykernel_12100/1099015402.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


,id,company_name_desc,content
0,142,Earbits — Google AdWords for music. Allowing ...,Google AdWords for music. Allowing bands and l...
1,267,Heyzap — Heyzap (acquired by Fyber for $45m in...,Heyzap is a premier mobile ad network that hel...
2,28921,Surface Labs — Building the AI-powered marketi...,"Surface Labs helps B2B companies capture, qual..."
3,451,Credictive — Attribution metatags for online c...,"Industries: B2B, Engineering, Product and Desi..."
4,27906,1stCollab — Performance-Optimized Influencer M...,1stCollab is the first influencer platform tha...


##### Save/Load Sample Questions

In [4]:
# df.to_csv('data/samples_company_records.csv')

df = pd.read_csv('data/samples_company_records.csv')

In [5]:
company_records = df.to_dict(orient='records')

In [6]:
company_records[:10]

[{'index': 0,
  'id': 1754,
  'company_name_desc': 'Mighty Buildings — 3D printing beautiful, high-quality, and sustainable homes.',
  'content': "Mighty Buildings is an innovative construction technology company based in Oakland, CA creating beautiful, sustainable, and high-quality homes using 3d-printing, robotics, and automation. Their mission is to have a positive impact on the environment, local communities, and the housing crisis through their sustainable approach.\n\nMighty Buildings' technology has the potential to unlock the needed productivity for large scale construction alongside the opportunity for reduced emissions, leading to a more sustainable product and future. Mighty Buildings was founded by a team of physicists and robotics engineers with extensive experience solving hard R&D problems and building successful engineering firms. They started by inventing a new material that is a 3D printing tech that enabled printing of an entire building, not just walls, in a single 

#### Generating Ground Truth

In [6]:
prompt_template = """
You emulate an MBA student researching on businesses.
Formulate 5 questions this student might ask based on a company record.
If the content contains the company name, or any identifier to the company, remove the company name/identifier. The questions should NOT contain company identifier (names etc). 
The questions should like a leading question looking at the company description, with a generalized tone
The questions should be complete and neither too short nor too long. If possible, use as fewer words as possible from the record. 

content: {content}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

# output the questions only in the parsable JSON format ["question1", "question2", ..., "question5"]; don't use code blocks


In [7]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response


In [8]:
def generate_questions_ollama(doc):
    prompt = prompt_template.format(**doc)
    prompt += 'ensure to output the questions only in the parsable JSON format ["question1", "question2", ..., "question5"], and that the array format is closed'
    # prompt += 'remove code blocks from output'

    response = ollama.chat(
        model='llama3.2', #'gemma3', #'mistral', #'llama3.2',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.message.content
    return json_response

In [ ]:
from concurrent.futures import ThreadPoolExecutor

results_openai = {}
results_llama = {}

with ThreadPoolExecutor(max_workers=2) as executor:
    for doc in tqdm(company_records):
        doc_id = doc['id']
        if doc_id in results_openai and doc_id in results_llama:
            continue

        future_openai = executor.submit(generate_questions, doc)
        future_llama = executor.submit(generate_questions_ollama, doc)

        results_openai[doc_id] = future_openai.result()
        results_llama[doc_id] = future_llama.result()

  0%|          | 0/524 [00:00<?, ?it/s]

In [9]:
import json
# print(f'openai: {results_openai[1754]}\nollama: {results_llama[1754]}')

# json.loads(results_openai[1754])

print(results_llama[1754])
json.loads(results_llama[1754])

NameError: name 'results_llama' is not defined

In [23]:
# type(results_openai)
# type(results_llama)

#### Save ground truths

In [44]:
import json

def to_rows(results, test=False):
    rows = []
    for (k, v) in results.items():
        try:
            ques = json.loads(v)
            for q in ques:
                rows.append({
                    "company_id": k, "question": q
                })
        except json.JSONDecodeError:
            if test:
                print(f"Skipping company_id {k}: could not parse JSON: {v!r}")
                
            continue

     
    return None if test else rows

##### Test parsing

In [45]:
to_rows(results_openai, True)

In [46]:
to_rows(results_llama, True)

Skipping company_id 25733: could not parse JSON: '[\n  "How does this software help solar utilities predict and prevent downtime disruptions, and what specific benefits do they receive?",\n  "What data points are being used to train the algorithms in the platform, and how accurate are the predictions?",\n  "Can the platform provide real-time monitoring of solar plant performance, and if so, what kind of insights can it offer?",\n  "How does the software help reduce costs for solar utilities, and what specific cost savings have been achieved?",\n  "What kind of scalability and reliability can the cloud-based platform expect to deliver, given its large-scale data processing capabilities?"'
Skipping company_id 27708: could not parse JSON: '[\n  "What was the primary focus of this business, given its B2B subindustry classification?",\n  "How did the use of AI technology impact the company\'s operations or strategy?",\n  "In what ways did the company plan to expand or grow in its early stag

##### Saving CSVs

In [ ]:
# openai_rows = to_rows(results_openai)
# ollama_rows = to_rows(results_llama)

# if len(openai_rows) > 0:
#     ground_truth_df = pd.DataFrame(openai_rows)
#     ground_truth_df.to_csv('data/ground_truths_by_openai.csv')

# if len(ollama_rows) > 0:
#     ground_truth_ollama_df = pd.DataFrame(ollama_rows)
#     ground_truth_ollama_df.to_csv('data/ground_truths_by_ollama.csv')


#### Load ground truths

In [7]:
ground_truth_openaidf = pd.read_csv('data/ground_truths_by_openai.csv')
ground_truth_ollmadf = pd.read_csv('data/ground_truths_by_ollama.csv')
# ground_truth_df.head()

#### Hit-rates and MRR

In [8]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

from tqdm.auto import tqdm
import importlib
import rag_helper

importlib.reload(rag_helper)
from rag_helper import RAGBase

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

sentence_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

[nltk_data] Downloading package stopwords to /Users/Daudi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
class RAGPgVector(RAGBase):

    def __init__(self, embedder, conn, **kwargs):
        super().__init__(index=None, **kwargs)
        self.embedder = embedder
        self.conn = conn

    def vec_to_str(self, vector):
        return '[' + ','.join(str(x) for x in vector) + ']'

    def text_search(self, query, num_results=5):
        words = word_tokenize(query)
        stop_words = set(stopwords.words('english'))
        filtered_keywords = [w for w in words if w.isalnum() and w.lower() not in stop_words]

        sql = """
            SELECT
                    company_id,
                    title,
                    content,
                    ts_rank(search_vector, query) AS rank
            FROM
                    yc_oss_fulltext,
                    websearch_to_tsquery('english', '{wapi}') AS query
            WHERE search_vector @@ query
            ORDER BY rank desc
            limit {limits}
        """.format(wapi=' or '.join(filtered_keywords), limits=num_results)

        rows = self.conn.execute(sql).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'match_words_frequency_rank': r[3]}
            for r in rows
        ]
        

    def vector_search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        query_str = self.vec_to_str(query_vector)

        rows = self.conn.execute(
            """
            SELECT 
                company_id,
                title,
                content,
                1 - (embedding <=> %s::vector) AS cosine_similarity
            FROM yc_oss_embeddings
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_str, query_str, num_results)
        ).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'cosine_similarity': r[3]}
            for r in rows
        ]

##### Relevance functions

In [10]:

def compute_single_relevance(q: dict, search_func):
    doc_id = q["company_id"]
    results = search_func(query=q["question"])

    relevance = []
    for d in results:
        if d["company_id"] == doc_id:
            # relevance.append(int(d["company_id"] == doc_id))
            if 1 in relevance: 
                relevance.append(0)
            else:
                relevance.append(1)
        else:
            relevance.append(0)


    return relevance

def compute_all_relevance(ground_truth: dict, search_func):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_single_relevance(q, search_func)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance: list[list[int]]):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance: list[list[int]]):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


##### Search

In [11]:
pconn = psycopg.connect(
        host="localhost",
        port=5440,
        dbname="ventures_db",
        user="postgres",
        password="postgres",
    )

pgIndex = RAGPgVector(
    embedder=sentence_transformer_model,
    conn=pconn,
    llm_client=openai_client,
    # prompt_template=prompt_template,
)

pgIndex_local = RAGPgVector(
    embedder=sentence_transformer_model,
    conn=pconn,
    llm_client=ollama,
    # prompt_template=prompt_template,
)

In [12]:
pgIndex.text_search('companies building proprietary material')


[{'company_id': 24291,
  'title': 'DigiBuild — We buy & track building materials for construction companies.',
  'content': "Our platform fixes construction's broken supply chain. 70% of construction projects finish late or over budget; procuring and managing building materials is the number 1 reason why. \r\n\r\nWe use LLMs to buy and manage building materials for large construction companies.\r\nWith DigiBuild, construction contractors can find, order, track and manage their materials from the supplier to the job site. We help customers save money, staff time, and improve construction project schedules.\r\n\r\nWe work with the top real estate developers, general contractors, and subcontractors in the US. We manage billions in building material volume annually in the 2nd largest market in the world.\nIndustries: Real Estate and Construction, Construction | Subindustry: Real Estate and Construction -> Construction | Tags: Artificial Intelligence, SaaS, Supply Chain, AI, ML\nLocation: M

In [15]:
pgIndex.vector_search('companies building proprietary material', 5)

# pgIndex.vector_search("""
#     How does the company's innovative technology significantly 
#     alter the traditional construction processes to address sustainability challenges?
# """, 3)


[{'company_id': 30353,
  'title': 'Axal — Service company that designs, sources, and quality-tests custom PCBs',
  'content': 'We help companies get custom PCBs built without having to manage multiple vendors themselves. We review or create PCB designs, find the right manufacturer, coordinate production, inspect and test every board, handle any manufacturing issues, and deliver the finished boards directly to your office.\r\n\r\nOur goal is to reduce manufacturing errors, shorten iteration cycles, and get your boards to you as quickly as possible without compromising on quality.\nIndustries: Industrials, Manufacturing and Ro',
  'cosine_similarity': 0.5200363707564107},
 {'company_id': 25810,
  'title': "Material Depot — India's Fastest Growing Home Decor destination for Floor & Wall Decor",
  'content': 'rapidly growing digital-first ecosystem, Material Depot isn’t just selling materials—we’re building the future of how India designs and lives.\nIndustries: Real Estate and Constructio

###### compute_single_relevance

In [21]:
qObj = {
        'company_id': 1754,
        # 'question': "How does the company's innovative technology significantly alter the traditional construction processes to address sustainability challenges?"
        'question': 'companies building proprietary material'
    }

res = compute_single_relevance(qObj, pgIndex.text_search)
# res = compute_single_relevance(qObj, lambda query=qObj, n=10: pgIndex.text_search(query,n))
res

[0, 0, 1, 0, 0]

In [22]:
qObj = {
        'company_id': 1754,
        'question': "How does the company's innovative technology significantly alter the traditional construction processes to address sustainability challenges?"
        # 'question': 'companies building proprietary material'
    }

res = compute_single_relevance(qObj, pgIndex.vector_search)
# res = compute_single_relevance(qObj, lambda query=qObj, nr=10: pgIndex.vector_search(query,nr))
res

[1, 0, 0, 0, 0]

#### Evaluations (Actuals)

###### openai

In [34]:
openai_text_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.text_search)
openai_vector_relevances = compute_all_relevance(ground_truth_openaidf.to_dict(orient='records'), pgIndex.vector_search)

openai_hit_rate_txt = hit_rate(openai_text_relevances)
openai_hit_rate_vec = hit_rate(openai_vector_relevances)

openai_mrr_txt = mrr(openai_text_relevances)
openai_mrr_vec = mrr(openai_vector_relevances)

print(f""""
    Hit rates: Text search: {openai_hit_rate_txt}, Vector search: {openai_hit_rate_vec}\n
    MRR: Text search: {openai_mrr_txt}, Vector search: {openai_mrr_vec}
""")


  0%|          | 0/2620 [00:00<?, ?it/s]

  0%|          | 0/2620 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.2820610687022901, Vector search: 0.2824427480916031

    MRR: Text search: 0.20036259541984686, Vector search: 0.21107506361323133



###### ollama

In [35]:
ollama_text_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.text_search)
ollama_vector_relevances = compute_all_relevance(ground_truth_ollmadf.to_dict(orient='records'), pgIndex.vector_search)

ollama_hit_rate_txt = hit_rate(ollama_text_relevances)
ollama_hit_rate_vec = hit_rate(ollama_vector_relevances)

ollama_mrr_txt = mrr(ollama_text_relevances)
ollama_mrr_vec = mrr(ollama_vector_relevances)

print(f""""
    Hit rates: Text search: {ollama_hit_rate_txt}, Vector search: {ollama_hit_rate_vec}\n
    MRR: Text search: {ollama_mrr_txt}, Vector search: {ollama_mrr_vec}
""")


  0%|          | 0/2487 [00:00<?, ?it/s]

  0%|          | 0/2487 [00:00<?, ?it/s]

"
    Hit rates: Text search: 0.27342179332529154, Vector search: 0.22798552472858866

    MRR: Text search: 0.21337622302640386, Vector search: 0.17066076933386926



> #### Hit Rate/MRR Interpretation
> This results are as expected because of the reasons below:
>   - The questions are built from a subset of the database (1/10th), therefore other results can fit better
>   - Testing on the main data includes other similar companies that answer the questions in the ground truth
>   - The data we're looking at is more generalized than, for example, specific company documents with specific information
> <br/>
> &nbsp;

##### RRF

In [13]:
def rrf(result_lists: list[list[dict]], k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = doc["company_id"]
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query: str, num_recs = 5):
    text_results = pgIndex.text_search(query, num_results=num_recs)
    vector_results = pgIndex.vector_search(query, num_results=num_recs)

    return rrf([text_results, vector_results], num_results=num_recs)



In [14]:
query = 'companies building proprietary material'

res = hybrid_search(query)

res


[{'company_id': 25810,
  'title': "Material Depot — India's Fastest Growing Home Decor destination for Floor & Wall Decor",
  'content': 'Material Depot is reimagining home interiors for the next generation of Indian homeowners. We bring design-forward materials: tiles, laminates, wall panels & more - straight from manufacturers to customers, cutting out the old, broken supply chain. Our online engagement fuels real-time trend insights, enabling us to launch new designs faster, in smaller batches, and at unbeatable prices. The result: beautiful homes built smarter, quicker, and more affordably. With over 10,000 homes served and a ',
  'cosine_similarity': 0.46236670017242765},
 {'company_id': 24291,
  'title': 'DigiBuild — We buy & track building materials for construction companies.',
  'content': "Our platform fixes construction's broken supply chain. 70% of construction projects finish late or over budget; procuring and managing building materials is the number 1 reason why. \r\n\r\

#### LLM Search

In [15]:
pgIndex.rag(query, res)

'Fair sir, from the scroll I spy but one true fit for **companies building proprietary material**:\n\n**Mighty Buildings** —  \nLo, they didst invent a wondrous matter of their own,  \na new-made substance for the printing of whole homes,  \nlighter than concrete, and fashion’d for the hand of robots.  \nNot mere walls alone, but dwellings in a single cycle  \nthey bringeth forth with craft and cunning.  \nThus doth their house-borne alchemy stand foremost in this quest.\n\nIf thou wouldst ask further, try these queries:\n- companies inventing proprietary materials for construction\n- startups with custom material science for homes\n- companies making novel building materials\n- proprietary robotics materials for manufacturing\n- companies developing new construction tech materials'

#### Agentic Search

In [19]:
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass
import logfire

@dataclass
class SearchDeps:
    index: RAGPgVector

yc_startup_agent = Agent(
    'openai:gpt-5.4-mini',
    deps_type=SearchDeps,
    instructions=pgIndex.instructions,
)

@yc_startup_agent.tool
def search(ctx: RunContext[SearchDeps], query: str) -> str:
    # ctx.deps.index is the minsearch index we injected via SearchDeps
    res = hybrid_search(query)
    return res

logfire.configure()
logfire.instrument_pydantic_ai()

deps = SearchDeps(index=pgIndex_local)
result = await yc_startup_agent.run('companies building proprietary material', deps=deps)

result.output

17:47:22.565 yc_startup_agent run
17:47:22.567   chat gpt-5.4-mini


Logfire project URL: https://logfire-eu.pydantic.dev/dakn2025/starter-project

17:47:23.937   running tool: search
17:47:24.288   chat gpt-5.4-mini


'I haven’t found a company satisfying the query,  \nFor “proprietary material,” none doth fully sing;  \nYet nearby stars in this gathered company are these:\n\n**Metal** — a platform of private-captial lore,  \nWhose craft is to forge firm-knowing into firm-winning power.  \nWith **proprietary intelligence** on venture’s realm,  \nIt guides founders to the likely backers at the helm.\n\n**Storia AI** — a copilot bold and keen,  \nThat knows thy company’s code and context unseen.  \nThough not of material wrought, its mind is deep and rare,  \nA proprietary understanding of systems it doth bear.\n\n**DigiBuild** — not materials of own design,  \nBut building materials it buys and tracks in line.  \nIt serves the construction trade with supply-chain art,  \nYet proprietary matter is not its central part.\n\n**Minro** — a keeper of user knowledge bright,  \nWith contextual intelligence to bring churn into sight.  \nIt shapes customer wisdom, yet not material’s throne;  \nSo here it stand